# HW6: Lasso Regression - Guide



# Problem 1: LASSO Regression

We consider the Lasso estimator:

$$
\hat{\beta}
=
\arg\min_{\beta \in \mathbb{R}^d}
\left\{
\frac{1}{n}\|Y - X\beta\|_2^2 + \lambda \|\beta\|_1
\right\}.
$$

Our first task is to generate synthetic data and implement the `SparseLinearRegressionDataset` class that will be used for training.

## Data Generation Model

You can generate the data easily using built-in PyTorch functions.

### Mathematical Specification

Generate the data $Y = Xw + \epsilon$ where:

**Feature matrix:** $X \in \mathbb{R}^{n \times d}$
- Each entry: $X_{ij} \sim \mathcal{N}(0, 1)$ (independent standard normal)
- $n$ = number of samples, $d$ = input dimension

**True sparse weight vector:** $w \in \mathbb{R}^d$
- $w_j = 1$ for $j \leq \lfloor d \times \text{sparsity} \rfloor$ (the first $s$ entries are 1)
- $w_j = 0$ for $j > \lfloor d \times \text{sparsity} \rfloor$ (the rest are 0)

**Noise:** $\epsilon \sim \mathcal{N}(0, \sigma^2 I_n)$
- Independent Gaussian noise with standard deviation $\sigma = \text{noise\_std}$

**Response:** $Y = Xw + \epsilon$
- Linear combination of features plus noise

## Key Concepts for PyTorch Dataset

### The Dataset Class Structure

The class `SparseLinearRegressionDataset` inherits from the class `torch.utils.data.Dataset`. We need to initialize these essential methods

1. **`__init__`**: Initialize the dataset
   - Generate or load the data
   - Store references to features and labels
   - Perform any preprocessing needed

2. **`__len__`**: Return the total number of samples
   - Used by DataLoader to know how many samples exist
   - Enables efficient iteration and batching

3. **`__getitem__(idx)`**: Return a single sample by index
   - Called by DataLoader to fetch samples during training
   - Returns a tuple `(feature, label)` for the sample at position `idx`



## Complete Implementation

Below is the complete `SparseLinearRegressionDataset` class that should be placed in `src/dataset.py`:

In [ ]:
# src/dataset.py
import torch
import os
from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class SparseLinearRegressionDataset(Dataset):    
    def __init__(self, num_samples=100, input_dim=100, sparsity=0.5, noise_std=1):
        # Generate feature matrix X ~ N(0, 1) with shape (num_samples, input_dim)
        self.x = torch.randn(num_samples, input_dim)
        
        # Create sparse weight vector w
        w = torch.zeros(input_dim)
        num_nonzero = int(input_dim * sparsity)  # Number of non-zero entries
        w[:num_nonzero] = 1  # Set first num_nonzero entries to 1
        
        # Generate response: Y = X @ w + noise
        # X @ w: matrix-vector multiplication
        # noise: shape (num_samples,) from N(0, noise_std^2)
        noise = torch.randn(num_samples) * noise_std
        self.y = self.x @ w + noise
        
        # Store metadata
        self.num_samples = num_samples
        self.input_dim = input_dim
        self.sparsity = sparsity
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

## Usage Example - DataLoaders

Once you've implemented the `SparseLinearRegressionDataset` class in `src/dataset.py`, you can use it as follows:

A `DataLoader` takes a `Dataset` instance and handles batching, shuffling, and iteration.

We just made an instance of `SparseLinearRegressionDataset`, which is a child of `Dataset`.


In [3]:
# Example usage (once implementation is complete)

import torch
from torch.utils.data import DataLoader

# from src.dataset import SparseLinearRegressionDataset  
# In your homework, you would import from src.dataset

# Create a dataset with specific hyperparameters
train_dataset = SparseLinearRegressionDataset(
    num_samples=1000,   # n: number of training samples
    input_dim=500,      # d: feature dimension
    sparsity=0.1,
    noise_std=0.1
)

# Create a DataLoader for batching
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Iterate through batches
for batch_x, batch_y in train_loader:
    # batch_x shape: (32, 500) → 32 samples, each with 500 features
    # batch_y shape: (32,)     → 32 response values

    print(f"Batch X shape: {batch_x.shape}")
    print(f"Batch Y shape: {batch_y.shape}")
    break  # Just show the first batch

Batch X shape: torch.Size([32, 500])
Batch Y shape: torch.Size([32])


## Problem 1.2 Implement Loss Function

$$\frac{1}{n}||y-\hat y||_2^2 + \lambda||w||_1.$$




In [6]:
# The nn.MSELoss() is useful here
y_true_example = batch_y  # True response values
y_pred_example = torch.randn(32)  # Example predictions
criterion = torch.nn.MSELoss()
loss = criterion(y_true_example, y_pred_example)
print(f"Example Loss: {loss.item()}")

# L1 norm can also be implemented using torch.norm with p=1
w_example = torch.randn(500)  # Example weight vector
l1_norm = torch.norm(w_example, p=1)
print("Example L1 Norm of w:", l1_norm.item())

Example Loss: 73.09022521972656
Example L1 Norm of w: 403.5696105957031


## Problem 1.3 Define a new optimizer

See additional notes about the actual `Iterative Shrinkage-Thresholding Algorithm (ISTA)` and its accelerated extension, the `FISTA`

In `src/train.py`, your incomplete `ISTA` class inherits from the `Optimizer` class in `torch.optim.optimizer`

In [8]:
from torch.optim.optimizer import Optimizer

class ISTA(Optimizer):
    def __init__(self, params, lr=1e-3, l1_reg=1e-3):
        """
        Args:
            params (iterable): Iterable of parameters to optimize.
            lr (float): Learning rate.
            l1_reg (float): L1 regularization strength.
        """
        defaults = dict(lr=lr, l1_reg=l1_reg)
        super(ISTA, self).__init__(params, defaults)

You can look in the pdf document for this lab. The math doesn't look too hard, but the implementation can be challenging. 

In the `train.py` file, you will find three classes
- `ISTA`, inherits from Optimizer
- `FISTA`, inherits from Optimizer
- `trainer`, a helper class that is initialized with
    - model
    - train_loader (DataLoader)
    - val_loader
    - loss function
    - optimizer
    - and many more

You can read more about the `Optimizer` class in PyTorch:
https://docs.pytorch.org/docs/stable/optim.html

In PyTorch, you pair an `Optimizer` with a `Model`. In this homework, your models are defined in `model.py`, such as `LinearRegressionModel` and `NonConvexModel`.

When you build an `Optimizer`, you pass in `model.parameters()` so the optimizer knows which parameters to update.

Example:
`optimizer = ISTA(model.parameters(), lr=1e-3)`

## For Problem 1.3

`self.param_groups` is a list of dictionaries. Each dictionary stores both:

- model parameters (what to update)
- optimizer hyperparameters (how to update them)

For example:
```
{
    'params': [p1, p2, ...],   # model parameters (weights)
    'lr': 0.001,               # learning rate
    'l1_reg': 0.001,           # regularization strength
    'lambda_t': 1.0            # (FISTA only) momentum scalar
}
```

In [ ]:
# train.py
import math
import torch
import torch.nn as nn
from torch.optim.optimizer import Optimizer
import datetime
import subprocess
import os
import sys
from utils import *
from tqdm import tqdm

class ISTA(Optimizer):
    # params are the model parameters to optimize
    def __init__(self, params, lr=1e-3, l1_reg=1e-3):
        """
        Args:
            params (iterable): Iterable of parameters to optimize.
            lr (float): Learning rate.
            l1_reg (float): L1 regularization strength.
        """
        defaults = dict(lr=lr, l1_reg=l1_reg)

        # super() calls the parent Optimizer class constructor
        # This initializes important internal structures like:
        # - self.param_groups: stores parameters + hyperparameters
        # - self.state: stores per-parameter state (used in optimizers like FISTA)
        # It takes the model parameters and optimizer hyperparameters (defaults)
        # and organizes them into self.param_groups
        super().__init__(params, defaults)

    def step(self):
        """
        Performs a single optimization step.
        """
        for group in self.param_groups:
            # Usually there's only one group, but PyTorch allows multiple groups with different hyperparameters

            # For each group, extract hyperparameters
            lr = group['lr']
            l1_reg = group['l1_reg']
            
            # Turn off gradient tracking, since we're manually updating parameters

            

            with torch.no_grad():
                # Note parms are vectors
                for param in group['params']:
                    if param.grad is None:
                        continue
                    # Perform a gradient descent step:
                    z = param - lr * param.grad
                    # Apply soft-thresholding (proximal operator for L1)
                    # Soft thresholding: S(x, alpha) = sign(x) * max(|x| - alpha, 0)


                    # Update
                    
                    # Override the parameter with the new value after soft-thresholding

                    # Don't do param = ... because that would just rebind the local variable param, not update the actual parameter tensor in-place.

                    param.copy_(torch.sign(z) * torch.clamp(torch.abs(z) - lr * l1_reg, min=0.0))


#TODO: Problem 1.3: Complete the FISTA optimizer following the instructions and hints inside the class
class FISTA(Optimizer):
    def __init__(self, params, lr=1e-3, l1_reg=1e-3):
        """
        Args:
            params (iterable): Iterable of parameters to optimize.
            lr (float): Learning rate.
            l1_reg (float): L1 regularization strength.
        """
        # Initialize the defaults
        defaults = dict(lr=lr, l1_reg=l1_reg, lambda_t=1.0)
        super(FISTA, self).__init__(params, defaults)

    def step(self):
        """
        Performs a single FISTA optimization step.
        """

        for group in self.param_groups:
            """
            Hint:group is a dictionary containing optimizer parameters and their values
            As in __init__, we initialize the defaults for the optimizer parameters
            super(FISTA, self).__init__(params, defaults)
            So each group contains:
            {
                'params': [parameter tensors],  # List of model parameters to optimize
                'lr': learning_rate,           # Learning rate for this parameter group
                'l1_reg': l1_regularization,   # L1 regularization strength
                'lambda_t': momentum_term      # FISTA-specific momentum scalar
            }
            """
            lr = group['lr']
            l1_reg = group['l1_reg']
            # Retrieve the old momentum scalar
            lambda_old = group['lambda_t'] 
            #TODO: Compute the new momentum parameter lambda_new:  \lambda_{t+1} = \frac{1 + \sqrt{1+4\lambda_{t}^2}}{2} and update and update group['lambda_t'] as lambda_new

            with torch.no_grad():
                for p in group['params']:
                    if p.grad is None:
                        continue

                    """
                    Hint: self.state[p] is a per-parameter state dictionary in PyTorch optimizers
                    It's used to store and track parameter-specific values across optimization steps
                    self.state[p] = {
                        'momentum': ...,
                        'step': ...,
                        'x': ...   # <- you can define this key yourself
                    }
                    Each parameter p has its own state dict accessed via self.state[p]
                    Here we use it to store the previous x iterate ('x') for momentum calculations x_{t}
                    This state persists between steps, allowing us to implement stateful optimizers
                    """

                    # We use self.state[p]['x'] to save the auxiliary x_t for momentum calculations
                    if 'x' not in self.state[p]:
                        self.state[p]['x'] = p.data.clone() # x0 = y0 in initial step

                    #TODO: Add Gradient descent step: z = y_t -\eta \nabla f(y_t)

                    #TODO: Add Proximal Step by soft-thresholding: x_{t+1} = Soft-Threshold(z, l1_reg * lr)

                    #TODO: Momentum (acceleration) update: y_{t+1} = x_{t+1} + ((lambda_old - 1) / lambda_{t+1}) * (x_{t+1} - x_t)

                    #TODO: Update parameter p, save the new x
                    """
                    Hint: Use in-place operations like .copy_ and .clone() for memory efficiency and to maintain 
                    proper gradient tracking. 
                    """



## 2.1. Overriding

The last thing you should know how to do for this homework is some basic OOP. I gave you a `Trainer` class, suitable for a regression problem. In problem 2, you will deal with a more complicated objective function. You want to write a new trainer class `NonConvexTrainer` that inherits from the old `Trainer`. But some methods need to be changed, or `override`

`kwargs` and `args`

Previously, the `Trainer` Class contained many arguments: model, train_loader,...,l1_reg.

`args` is a convienient way to collect *positional* arguments, `kwargs` collect arguments like `a=b`

In [15]:
# simple example 
def f(*args):
    print(args)
f(1,2,3)
def g(*args,**kwargs):
    print(args)
    print(kwargs)
g(5,a=1, b=2, c=3)

(1, 2, 3)
(5,)
{'a': 1, 'b': 2, 'c': 3}


Why's are `kwargs` and `args` useful here? That way, we don't have to write 

```
def __init__(self, model, train_loader,...,)
```
in our constructor for the new class (NonconvexTrainer) that inherits from the old. 

In [ ]:
# main.py
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import random
import numpy as np
from model import NonConvexModel, nonconvex_objective
from train import Trainer
from config_nonconvex import *
from utils import plot_parameter_trajectory_on_contour

#TODO: Problem 2.1 Define a new trainer class inheriting from `Trainer` for nonconvex optimization
class NonconvexTrainer(Trainer):
    """
    A custom trainer that overrides the _evaluate_model method to directly 
    use the nonconvex objective function rather than using validation data.
    """
    def __init__(self, *args, scheduler=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Initialize list to track parameter values during training
        self.parameter_history = []
        self.scheduler = scheduler
        
    def _evaluate_model(self, pbar=None):
        """
        Override the evaluation method to directly compute the nonconvex objective
        without using validation data.
        """
        self.model.eval()  # Set the model to evaluation mode
        
        with torch.no_grad():
            #TODO: Update the objective_value by evaluating nonconvex_objective at self.model.a            
            
            # Log the objective value
            if pbar:
                pbar.set_postfix(objective=f"{objective_value.item():.6f}")
            
            # Return the objective value as the validation loss
            return objective_value.item()
    
    def _train_epoch(self, epoch, pbar=None):
        """Train the model for one epoch."""
        self.model.train()  # Set the model to training mode
        
        # Store current parameter value before training step
        self.parameter_history.append(self.model.a.clone().detach())
        
        current_loss = nonconvex_objective(self.model.a)
        running_loss = current_loss.item()

        if self.hyperparams['OPTIMIZER'] != 'GD':
            # Here for stochastic optimizers, we want to add noise zeta ~ N(0, 1/sqrt(batch_size)) to the gradient
            # Instead of rewrite the optimizer, we can revise the loss function f(a) -> f(a) + zeta * sum(a) 
            # then the gradient will be grad(f(a)) + zeta
            zeta = torch.randn((), device=self.device) / (self.hyperparams['BATCH_SIZE'] ** 0.5)
            current_loss += torch.sum(self.model.a) * zeta
        
        #TODO: Update the model parameters using the optimizer


        # Step the scheduler if it exists
        if self.scheduler:
            self.scheduler.step()

        # Update progress bar with current loss if provided
        if pbar:
            pbar.update(1)
            pbar.set_postfix(train_loss=f"{current_loss.item():.3f}")

        return running_loss
    
    def train(self):
        """Override the train method to add parameter trajectory visualization"""
        #TODO: Call the parent class's train method
        
        #TODO: Plot the parameter trajectory on the contour using `plot_parameter_trajectory_on_contour` from `utils.py`

